In [ ]:
import pandas as pd
from datetime import datetime
from IPython.display import display
import os
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv('/content/CSAW-CC_breast_cancer_screening_data.csv')
df = df.drop(columns = ['x_cancer_laterality', 'x_type','x_age','x_lymphnode_met', 'rad_recall','rad_recall_type_left','x_case', 'rad_r1', 'rad_r2', 'rad_recall_type_right'])
display(df)

In [ ]:
cancer_years = {}

# Step 1: Identify cancer year for each patient
cancer_years = {}
for patient_id, group in df.groupby("anon_patientid"):
    # Get the most recent exam year for this patient
    most_recent_exam = group.loc[group['exam_year'].idxmax()]

    rad_timing = most_recent_exam['rad_timing']

    # Step 2: Calculate cancer year based on rad_timing
    if rad_timing == 1:  # Screen-detected cancer (within 60 days)
        cancer_years[patient_id] = most_recent_exam['exam_year']
    elif rad_timing == 2:  # Interval cancer (60-729 days)
        # If the last exam year is 2013, set cancer year to 2015
        cancer_years[patient_id] = most_recent_exam['exam_year'] +1
        if most_recent_exam['exam_year'] == 2015:
            cancer_years[patient_id] = 2016

# Step 3: Calculate years to cancer for each image
df["years_to_cancer"] = df.apply(lambda row: (
    cancer_years.get(row["anon_patientid"], None) - row["exam_year"]
    if row["anon_patientid"] in cancer_years else None
), axis=1)

# Step 4: Calculate years to last follow-up (based on the latest exam year for each patient)
latest_exam_years = df.groupby("anon_patientid")["exam_year"].max()
df["years_to_last_followup"] = df.apply(lambda row: (
    latest_exam_years[row["anon_patientid"]] - row["exam_year"]
), axis=1)

# Output the final DataFrame with calculated values

display(df)

In [ ]:
df_filtered = df.groupby('anon_patientid').filter(lambda x: x['exam_year'].nunique() > 1)
df_filtered['years_to_cancer'] = df_filtered['years_to_cancer'].fillna(100)


In [ ]:
# Step 1: Identify cancer-free patients with at least 5 years of follow-up
cancer_free_patients = df_filtered[df_filtered['years_to_cancer'] == 100]
follow_up_period = cancer_free_patients.groupby('anon_patientid')['exam_year'].agg(['min', 'max'])
follow_up_period['duration_years'] = follow_up_period['max'] - follow_up_period['min']

# Select patients with at least 5 years of follow-up
patients_with_5_years = follow_up_period[follow_up_period['duration_years'] >= 5].index
# Step 2: Identify patients with years_to_cancer in [1, 2, 3, 4, 5]
cancer_patients = df_filtered[df_filtered['years_to_cancer'].isin([0,1, 2, 3, 4, 5])]['anon_patientid'].unique()

# Step 3: Combine both groups of patients
valid_patients = set(patients_with_5_years).union(set(cancer_patients))

# Step 4: Apply filtering to keep only the selected patients
df_filtered_new = df_filtered[df_filtered['anon_patientid'].isin(valid_patients)].reset_index(drop=True)
csaw_cc_filtered = df_filtered_new[(df_filtered_new['years_to_cancer'] <= 5) | (df_filtered_new['years_to_cancer'] == 100)]

# Display the filtered dataset
display(csaw_cc_filtered)

In [ ]:
#csaw_cc_filtered.loc[csaw_cc_filtered['years_to_cancer'] == 0, 'years_to_cancer'] = 1

# Count the occurrences of each "years_to_cancer" value (2, 3, 4, 5)
counts = {    1: len(csaw_cc_filtered[csaw_cc_filtered['years_to_cancer'] == 1]) + len(csaw_cc_filtered[csaw_cc_filtered['years_to_cancer'] == 0]),
    2: len(csaw_cc_filtered[csaw_cc_filtered['years_to_cancer'] == 2]),
    3: len(csaw_cc_filtered[csaw_cc_filtered['years_to_cancer'] == 3]),
    4: len(csaw_cc_filtered[csaw_cc_filtered['years_to_cancer'] == 4]),
    5: len(csaw_cc_filtered[csaw_cc_filtered['years_to_cancer'] == 5]),
}

# Create a figure and axes for the plot
fig, ax = plt.subplots(figsize=(8, 6))

# Plot the bar chart with the counts for each year value (2Y, 3Y, 4Y, 5Y)
ax.bar(counts.keys(), counts.values(), edgecolor='black')

# Labeling the plot
ax.set_xlabel('Time to Cancer (Years to Diagnosis)')
ax.set_ylabel('Number of Images')
ax.set_title('Distribution of Time to Cancer')

# Add values above the bars
for i, value in enumerate(counts.values()):
    ax.text(list(counts.keys())[i], value + 10, str(value), ha='center', va='bottom', fontsize=10)

# Rotate x-axis labels for better readability
ax.tick_params(axis='x', rotation=45)

# Display the plot
plt.tight_layout()
plt.show()


In [ ]:

csaw_cc_filtered =  csaw_cc_filtered.rename(columns={'anon_patientid': 'patient_id', 'anon_filename': 'file_name', 'imagelaterality':'laterality'})
display(csaw_cc_filtered)

In [ ]:
csaw_cc_filtered['anon_dicom_path_local'] = '/storage/csaw_cc_data_full/2021-204-1-1/data/' + csaw_cc_filtered['file_name']
display(csaw_cc_filtered)

In [ ]:
csaw_cc_filtered['file_name'] = csaw_cc_filtered['file_name'].str.replace('.dcm', '.png', regex=False)
display(csaw_cc_filtered)

In [ ]:
from sklearn.model_selection import train_test_split

# Group the data by 'years_to_cancer'
grouped = csaw_cc_filtered.groupby('years_to_cancer')

# Empty lists to store splits

# First, we group the data by `patient_id` to ensure patients are only in one split
patients = csaw_cc_filtered['patient_id'].unique()

# Split the patients into train, validation, and test sets

train_patients, temp_patients = train_test_split(patients, test_size=0.5, stratify=csaw_cc_filtered.groupby('patient_id')['years_to_cancer'].first()[patients])
val_patients, test_patients = train_test_split(temp_patients, test_size=0.6, stratify=csaw_cc_filtered.groupby('patient_id')['years_to_cancer'].first()[temp_patients])

# Now, we can add the `split_group` to the original DataFrame based on patient assignment
csaw_cc_filtered['split_group'] = None
csaw_cc_filtered.loc[csaw_cc_filtered['patient_id'].isin(train_patients), 'split_group'] = 'train'
csaw_cc_filtered.loc[csaw_cc_filtered['patient_id'].isin(val_patients), 'split_group'] = 'val'
csaw_cc_filtered.loc[csaw_cc_filtered['patient_id'].isin(test_patients), 'split_group'] = 'test'

# Verify that the `split_group` is properly assigned and the distribution of `years_to_cancer` is preserved
print(csaw_cc_filtered.groupby(['split_group', 'years_to_cancer']).size())

# Verify the distribution and the new 'split_group' column


In [ ]:
bins = [0, 33, 67, 100]  # Bin edges
labels = ['0-33', '33-67', '67-100']  # Corresponding labels

# Create the 'density' column
csaw_cc_filtered['density'] = pd.cut(csaw_cc_filtered['libra_percentdensity'], bins=bins, labels=labels, include_lowest=True)
display(csaw_cc_filtered)

In [ ]:
csaw_cc_filtered.to_csv('/content/metadata_csawcc_dcm_path.csv', index=False)
